In [ ]:
#!/usr/bin/env python3
# trip_request_producer.py

import json, time, random, uuid, boto3, pyarrow.parquet as pq
import pyarrow.fs as fs
from kafka import KafkaProducer
from datetime import datetime
import re
from collections import defaultdict

# ========== CONFIGURATION ==========
KAFKA_BROKER   = "kafka:9092"
TOPIC_TRIPS    = "raw.trips"
MINIO_ENDPOINT = "http://100.104.4.32:9000"   # à ajuster
MINIO_KEY      = "minioadmin"
MINIO_SECRET   = "minioadmin123"
BUCKET         = "simulated"
PREFIX         = "porto-trajectories/"
SPEED_FACTOR   = 10

HOURLY_WEIGHTS = {
    0:0.15,1:0.10,2:0.08,3:0.06,4:0.07,5:0.12,6:0.30,7:0.75,8:1.00,
    9:0.85,10:0.65,11:0.60,12:0.55,13:0.50,14:0.55,15:0.65,16:0.80,
    17:1.00,18:0.95,19:0.75,20:0.55,21:0.45,22:0.35,23:0.25
}
FRIDAY_OVERRIDE = {12:0.20, 13:0.15, 14:0.25}

# ========== CONNEXION MINIO ==========
s3 = boto3.client("s3", endpoint_url=MINIO_ENDPOINT,
                  aws_access_key_id=MINIO_KEY, aws_secret_access_key=MINIO_SECRET)
response = s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX)
parquet_files = [obj["Key"] for obj in response.get("Contents", []) if obj["Key"].endswith(".parquet")]
print(f"📁 {len(parquet_files)} fichiers trouvés")

arrow_fs = fs.S3FileSystem(
    endpoint_override=MINIO_ENDPOINT,
    access_key=MINIO_KEY, secret_key=MINIO_SECRET, scheme="http"
)

# ========== COMPTER LES PAIRES ==========
pair_counts = defaultdict(int)
call_counts = defaultdict(int)
total = 0

for file_key in parquet_files:
    full_path = f"{BUCKET}/{file_key}"
    # Lire uniquement les colonnes dest_zone_id et CALL_TYPE
    try:
        table = pq.read_table(full_path, filesystem=arrow_fs,
                              columns=["dest_zone_id", "CALL_TYPE"])
    except Exception as e:
        print(f"Erreur lecture {file_key}: {e}")
        continue
    # Extraire origin_zone_id depuis le chemin
    match = re.search(r"origin_zone_id=(\d+)", file_key)
    if not match:
        continue
    origin = int(match.group(1))
    dest_list = table.column("dest_zone_id").to_pylist()
    call_list = table.column("CALL_TYPE").to_pylist()
    for d, c in zip(dest_list, call_list):
        pair_counts[(origin, d, c)] += 1
        call_counts[c] += 1
        total += 1

print(f"✅ {len(pair_counts)} paires uniques trouvées")
weighted_pairs = [(o, d, c, cnt/total) for (o, d, c), cnt in pair_counts.items()]
call_weights = {c: cnt/total for c, cnt in call_counts.items()}
print(f"   CALL_TYPE : {call_weights}")

# ========== KAFKA PRODUCER ==========
producer = KafkaProducer(
    bootstrap_servers=KAFKA_BROKER,
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    key_serializer=lambda k: str(k).encode("utf-8"),
    acks="all"
)
print("✅ Kafka connecté")

# ========== FONCTIONS ==========
def get_demand_weight():
    now = datetime.now()
    hour = now.hour
    w = HOURLY_WEIGHTS.get(hour, 0.5)
    if now.weekday() == 4 and hour in FRIDAY_OVERRIDE:
        w = FRIDAY_OVERRIDE[hour]
    if now.weekday() == 6:
        w *= 0.85
    return w

def choose_pair():
    r = random.random()
    cum = 0.0
    for o, d, c, prob in weighted_pairs:
        cum += prob
        if r <= cum:
            return o, d, c
    return weighted_pairs[-1][0], weighted_pairs[-1][1], weighted_pairs[-1][2]

# ========== BOUCLE PRINCIPALE ==========
print("\n📞 Trip Producer démarré")
count = 0
try:
    while True:
        weight = get_demand_weight()
        interval = (1.0 / max(weight, 0.1)) * (3 / SPEED_FACTOR)
        o, d, ct = choose_pair()
        event = {
            "trip_id": str(uuid.uuid4()),
            "rider_id": f"rider_{random.randint(1,50000):05d}",
            "origin_zone": o,
            "dest_zone": d,
            "requested_at": int(datetime.now().timestamp()),
            "call_type": ct,
            "demand_weight": round(weight, 2)
        }
        producer.send(TOPIC_TRIPS, key=event["trip_id"], value=event)
        count += 1
        if count % 10 == 0:
            print(f"   📍 {count} | heure={datetime.now().hour}h | poids={weight:.2f} | zone {o}→{d} ({ct})")
        time.sleep(interval)
except KeyboardInterrupt:
    print(f"\n⏹️ Arrêt — {count} réservations envoyées")
    producer.flush()
    producer.close()

📁 34 fichiers trouvés
✅ 596 paires uniques trouvées
   CALL_TYPE : {'C': 0.2759117569459948, 'A': 0.22087147119111886, 'B': 0.5032167718628864}
✅ Kafka connecté

📞 Trip Producer démarré
   📍 10 | heure=18h | poids=0.95 | zone 3→1 (B)
   📍 20 | heure=18h | poids=0.95 | zone 3→12 (C)
   📍 30 | heure=18h | poids=0.95 | zone 1→1 (B)
   📍 40 | heure=18h | poids=0.95 | zone 1→9 (A)
   📍 50 | heure=18h | poids=0.95 | zone 1→16 (C)
   📍 60 | heure=18h | poids=0.95 | zone 12→9 (B)
   📍 70 | heure=18h | poids=0.95 | zone 1→9 (C)
   📍 80 | heure=18h | poids=0.95 | zone 12→3 (B)
   📍 90 | heure=18h | poids=0.95 | zone 12→9 (B)
   📍 100 | heure=18h | poids=0.95 | zone 3→3 (B)
   📍 110 | heure=18h | poids=0.95 | zone 9→3 (B)
   📍 120 | heure=18h | poids=0.95 | zone 3→1 (C)
   📍 130 | heure=18h | poids=0.95 | zone 5→9 (A)
   📍 140 | heure=18h | poids=0.95 | zone 3→3 (C)
   📍 150 | heure=18h | poids=0.95 | zone 14→16 (C)
   📍 160 | heure=18h | poids=0.95 | zone 5→3 (C)
   📍 170 | heure=18h | poids=0.9

In [ ]:
trip.terminate()

In [ ]:
!ps aux | grep python

In [ ]:
gps.kill()
trip.kill()